### Embedding model form scrath for similarity search using the cosine similariy

Here is the full pipeline steps that we are going to follow,

- Text processing - Here we need to process the dataset from lowercaseing, strip punctuations and also remove noice.
- Tokenization - then we need to create the token ids from the text for this we can use the BPE
- Vocb and Enbedding layer -  Here we can create the vocab from the data and create the embeddings
- Encoder layer (Transformer layer) - A small transformer or MLP that we can add the contextual meaning.
- L2 Norm - From this we are going to project all vectors onto the unit hypersphere and this will make sure the cosine similarity will be equivalent to dot-product.
- Triplet Loss - This will help to get the similar tokens closer and apart the dissimilar ones.

### Step 01

Here we will load the dataset that we are gonna use, we are using the triplet sunset of the data where that contains,
- anchor
- positive
- negative

### The difference of an Embedding model and a Tokenizer

#### Tokenizer - Here a tokenizer will take a sentence and create token ids from that, see the example.

```
text :"Deep learning models are cool"

then this will be tokenized as
["Deep", " learning", " models", " are", " cool"]

then this will asign a tokenid
[5021, 1821, 993, 45, 812]

```

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch , torch.nn as nn , torch.nn.functional as F

# load the dataset
dataset  =  load_dataset("sentence-transformers/all-nli", "triplet", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

triplet/train-00000-of-00001.parquet:   0%|          | 0.00/38.4M [00:00<?, ?B/s]

triplet/dev-00000-of-00001.parquet:   0%|          | 0.00/782k [00:00<?, ?B/s]

triplet/test-00000-of-00001.parquet:   0%|          | 0.00/810k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/557850 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/6584 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6609 [00:00<?, ? examples/s]

In [3]:
dataset[0]

{'anchor': 'A person on a horse jumps over a broken down airplane.',
 'positive': 'A person is outdoors, on a horse.',
 'negative': 'A person is at a diner, ordering an omelette.'}

In [4]:
tokenizer  =  AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# Test the tokenizer
tokenizer(dataset[0]['anchor'], padding=True, truncation=True, return_tensors="pt")


{'input_ids': tensor([[   32,  1048,   319,   257,  8223, 18045,   625,   257,  5445,   866,
         19401,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

### Class for the Triplet Dataset

In [6]:
class  TripletDataset(torch.utils.data.Dataset):

    def __init__(self , data , tokenizer , max_length=32):
        super().__init__()
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self ):
        return len(self.data)
    
    def encode(self , text):
        return self.tokenizer(text , padding="max_length" , truncation=True , max_length=self.max_length , return_tensors="pt")["input_ids"].squeeze(0)
    
    def __getitem__(self , idx):
        anchor = self.data[idx]['anchor']
        positive = self.data[idx]['positive']
        negative = self.data[idx]['negative']

        return {
            "anchor": self.encode(anchor),
            "positive": self.encode(positive),
            "negative": self.encode(negative)
        }

In [7]:
# use the TripletDataset class
triplet_dataset = TripletDataset(dataset , tokenizer)
triplet_dataset[0]

{'anchor': tensor([   32,  1048,   319,   257,  8223, 18045,   625,   257,  5445,   866,
         19401,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256]),
 'positive': tensor([   32,  1048,   318, 24349,    11,   319,   257,  8223,    13, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256]),
 'negative': tensor([   32,  1048,   318,   379,   257, 47519,    11, 16216,   281,   267,
          1326, 21348,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256])}

### Model Architecture

Here are the different layers that are in the model architecture.

1. Embedding layer - When a token ID comes in, you just retrieve its row. That's it mechanically, but what makes it powerful is that these vectors are not fixed. They start random and get updated during training via backpropagation.

2. Transformer Encoder - The embedding layer gives each token an independent vector with no awareness of the other tokens around it. The Transformer encoder is what introduces context. It applies self-attention, which lets every token look at every other token in the sequence and decide how much to borrow from each one.

3. Mean Pooling -Mean pooling simply averages all the token vectors together into a single vector. This is deliberately simple. More complex options exist (like using a special [CLS] token the way BERT does) but mean pooling consistently performs well for short text and requires no additional learned parameters.

3. Projection - After pooling we apply one linear layer that maps from the encoder's dimension (256) down to the final output dimension (128). This serves two purposes. 

    -  First, it gives the model a learned bottleneck — a final compression step that forces the most important semantic signal to survive while discarding noise. 
 
    - Second, it decouples the internal working dimension from the output dimension, so you can tune them independently. A 128-dimensional output vector is fast to compare and store, while the 256-dimensional internal space gives the encoder room to work.

4. L2 Norm - The very last operation divides every output vector by its own magnitude, so every embedding lands on the surface of a unit hypersphere.

    -  cosine similarity between two normalized vectors equals their dot product. That means similarity search becomes a single matrix multiply with no division, which is dramatically faster at scale and compatible with optimized libraries like FAISS.

In [14]:
# variables
vocab_size = tokenizer.vocab_size
embedding_dim = 256
output_dim = 128
nhead = 4
num_layers = 2
ffn_dim = 512
dropout = 0.1

# for trainig
number_of_epochs = 3

In [20]:
class EmbeddingModel(nn.Module):

    def __init__(self , vocab_size , embedding_dim = embedding_dim , output_dim = output_dim , nhead = nhead , num_layers = num_layers , ffn_dim = ffn_dim , dropout = dropout):
        super().__init__()
        # embedding layer
        self.embedding = nn.Embedding(vocab_size , embedding_dim , padding_idx = 0)
        
        # transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model = embedding_dim,
            nhead = nhead,
            dim_feedforward = ffn_dim,
            dropout = dropout,
            batch_first=True
        )

        # binding the encoder layer to the transformer encoder
        self.encoder = nn.TransformerEncoder(
            encoder_layer = encoder_layer,
            num_layers = num_layers, # Here we specify the number of encoder layers in the transformer encoder
        )

        # Projection layer
        self.projection = nn.Linear(embedding_dim , output_dim)


    
    def forward(self , input_ids):
        x = self.embedding(input_ids)
        mask = (input_ids == 0)
        x = self.encoder(x , src_key_padding_mask =mask) # here the src_key_padding_mask enforces that attention scores for padding positions are set to negative infinity
        x = x.mean(dim=1)
        projected = self.projection(x)

        #  L2 normalization - for the cosine similarity loss function to work properly, we need to L2 normalize the output embeddings. Here each embedding vector is divided by its L2 norm, resulting in unit vectors. This ensures that the cosine similarity between two embeddings is equivalent to the dot product of the normalized vectors, which is crucial for the loss function to effectively measure the similarity between embeddings.
        return F.normalize(projected , p=2 , dim=1)

### Explaination on the faward function

1. create the embeddings usign the `self.embedding()` methode from nn. and this will give us a vector for the ton that has dim of (vocab_size , emb_dim). This will replace each token from the vector and then the output shape will go from (vocab_size ,no_tokens, emb_dim).

2. the mask - When you batch sequences together they must all be the same length, so shorter sequences get padded with zeros at the end. The token ID 0 is the padding token — it carries no real meaning.The mask is a boolean tensor — True wherever there's padding, False for real tokens. This gets passed to the encoder to tell it "ignore these positions completely."

3. The encoder - Through self-attention, each token looks at all other tokens in the sequence, computes relevance scores, and updates its own representation by borrowing information from neighbours. After the encoder, each token's vector now reflects its context.

### Training Loop

Here we are using **Triplet Loss** as the loss function. Each training sample consists of an **anchor**, a **positive**, and a **negative** example. The objective is to make the embedding of the anchor closer to the positive example while pushing the negative example farther away in the embedding space.

The loss function is defined as:

$$
L(a, p, n) = \max\left( d(a, p) - d(a, n) + \alpha, 0 \right)
$$

Where:

- $a$ = anchor embedding  
- $p$ = positive embedding (similar to anchor)  
- $n$ = negative embedding (dissimilar to anchor)  
- $d(\cdot)$ = distance function (commonly Euclidean or cosine distance)  
- $\alpha$ = margin that enforces separation between positive and negative pairs


In [13]:
def triplet_loss ( anchor , positive , negative , margin = 0.5):

    # cosine distance = 1 - cosine similarity (we have already L2 normalized the embeddings, so we can directly compute the cosine similarity as the dot product)
    pos_dist = 1 - (anchor * positive).sum(dim=1)
    neg_dist = 1 - (anchor * negative).sum(dim=1)
    loss = F.relu(neg_dist - pos_dist + margin)
    return loss.mean()

In [22]:
model = EmbeddingModel(vocab_size).cuda()
optimizer = torch.optim.Adam(model.parameters() , lr=1e-4)

loader = torch.utils.data.DataLoader(triplet_dataset , batch_size=64 , shuffle=True)


def train_model(model , loader , optimizer , number_of_epochs):

    for epoch in range(number_of_epochs):
        for batch in loader:

            # take the anchor , positive and negative sentences from the batch and move them to the GPU
            anchor = batch['anchor'].cuda()
            positive = batch['positive'].cuda()
            negative = batch['negative'].cuda()
            
            # pass the anchor , positive and negative sentences through the model to get their embeddings
            anchor_emb = model(anchor)
            positive_emb = model(positive)
            negative_emb = model(negative)

            # compute the triplet loss
            loss = triplet_loss(anchor_emb , positive_emb , negative_emb)

            # backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{number_of_epochs} - Loss: {loss.item():.4f}")

In [23]:
# GPU
!nvidia-smi

Sun Mar 15 05:21:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             26W /   70W |     623MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
train_model(model , loader , optimizer , number_of_epochs)

Epoch 1/3 - Loss: 0.3272
Epoch 2/3 - Loss: 0.2868
Epoch 3/3 - Loss: 0.3657


### Similarity Search

Once trained, we extract embeddings and use cosine similarity to find the most semantically related words in our corpus. Since vectors are L2-normalized, cosine similarity is just a dot product.

In [ ]:
# Encode a single string to an embedding vector.

def get_embedding(model , tokenizer , text , device = "cuda"):
    model.eval()
    with torch.no_grad():
        input_ids = tokenizer(text , padding="max_length" , truncation=True , max_length=32 , return_tensors="pt")["input_ids"].to(device)
        embedding = model(input_ids).squeeze(0)
    return embedding

In [26]:
# Return top-k most similar words from corpus."

def top_k_similar(query , corpus_embeddings , corpus_texts , k=5):

    sims = torch.matmul(query , corpus_embeddings)
    top_k_indices = torch.topk(sims , k=k)

    return [(corpus_texts[i] , sims[i].item()) for i in top_k_indices.indices]

In [27]:
# Test the model with some words from the corpus
# This will create a small corpus of words and compute their embeddings using the trained model. Then we can use the top_k_similar function to find the most similar words in the corpus for a given query word.

words = ["king", "queen", "man", "woman", "cat", "dog", "feline"]
corpus_embeddings = torch.stack([get_embedding(model , tokenizer , word) for word in words])

# query word
query_word = "cat"
query_embedding = get_embedding(model , tokenizer , query_word)

similar_words = top_k_similar(query_embedding , corpus_embeddings , words , k=3)
print(f"Top 3 similar words to '{query_word}':")
for word, similarity in similar_words:
    print(f"  {word}: {similarity:.4f}")

RuntimeError: Expected one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, maia, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string: GPU

### Save & Export the Model

Package the model for reuse: save weights, tokenizer config, and write a loading helper so the model can be used without the training code.

In [ ]:
import torch, json, os
from pathlib import Path

save_dir = Path("embedding_model_checkpoints")
save_dir.mkdir(exist_ok=True)

# 1. Save the model weights
torch.save(model.state_dict(), save_dir / "embedding_model_weights.pth")

# 2. Save the model configuration and hyperparameters
config = {
    "vocab_size": vocab_size,
    "embedding_dim": embedding_dim,
    "output_dim": output_dim,
    "nhead": nhead,
    "num_layers": num_layers,
    "ffn_dim": ffn_dim,
    "dropout": dropout,
    "max_length": 32,
}


with open(save_dir / "embedding_model_config.json", "w") as f:
    json.dump(config, f, indent=2)

# 3. Save the tokenizer
tokenizer.save_pretrained(save_dir / "tokenizer")

# 4. loading the model and tokenizer
def load_model(save_dir):
    # Load the model configuration
    with open(save_dir / "embedding_model_config.json", "r") as f:
        config = json.load(f)

    # Initialize the model with the loaded configuration
    model = EmbeddingModel(
        vocab_size=config["vocab_size"],
        embedding_dim=config["embedding_dim"],
        output_dim=config["output_dim"],
        nhead=config["nhead"],
        num_layers=config["num_layers"],
        ffn_dim=config["ffn_dim"],
        dropout=config["dropout"]
    )

    # Load the model weights
    model.load_state_dict(torch.load(save_dir / "embedding_model_weights.pth"))
    model.eval()  # Set the model to evaluation mode

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(save_dir / "tokenizer")

    return model, tokenizer


### Publish to Hugging Face Hub

Publishing makes your model discoverable and usable by anyone with huggingface_hub. Follow these steps in order.

In [ ]:
repo_id = "Saminx22/text_embedding_model_14M"
api = HfApi()

# 1. Create the repository
api.create_repo(repo_id=repo_id, exist_ok=True)

# 2. Create a detailed model description (README.md)
card_data = ModelCardData(
    language="en",
    license="apache-2.0",
    library_name="pytorch",
    model_name="14M Text Embedding Model",
    tags=["text-embeddings", "sentence-transformers", "triplet-loss"]
)

content = f"""
# 14M Text Embedding Model

This is a custom-built text embedding model trained from scratch using a Triplet Loss objective. 

### Model Architecture
- **Total Parameters:** {total_params:,}
- **Vocabulary Size:** 50,257 (GPT-2 Tokenizer)
- **Embedding Dimension:** 256
- **Output Dimension:** 128 (L2 Normalized)
- **Encoder:** 2-layer Transformer Encoder with 4 attention heads.

### Training
- **Dataset:** `sentence-transformers/all-nli` (Triplet subset)
- **Loss Function:** Triplet Loss with a margin of 0.5
- **Goal:** To project semantically similar sentences closer in a unit hypersphere, enabling efficient cosine similarity search.

### Usage
This model is designed for semantic similarity tasks where you need a lightweight but custom-trained embedding space.
"""

card = ModelCard.from_template(card_data, content=content)
card.save(save_dir / "README.md")

# 3. Upload the directory
api.upload_folder(
    folder_path=str(save_dir),
    repo_id=repo_id,
    repo_type="model"
)

print(f"Model successfully uploaded to: https://huggingface.co/{repo_id}")

### Demo in Huggingface Spaces

In [ ]:
import os
from huggingface_hub import HfApi

# Define the Space ID
space_id = "Saminx22/text-embedding-demo"
api = HfApi()

# 1. Create the Space repository (using Gradio SDK)
api.create_repo(repo_id=space_id, repo_type="space", space_sdk="gradio", exist_ok=True)

# 2. Define the app.py content using a raw string to avoid escaping issues
app_code = r"""
import gradio as gr
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
import json
from huggingface_hub import hf_hub_download

class EmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, output_dim=128, nhead=4, num_layers=2, ffn_dim=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=nhead, dim_feedforward=ffn_dim, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=num_layers)
        self.projection = nn.Linear(embedding_dim, output_dim)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        mask = (input_ids == 0)
        x = self.encoder(x, src_key_padding_mask=mask)
        x = x.mean(dim=1)
        projected = self.projection(x)
        return F.normalize(projected, p=2, dim=1)

REPO_ID = "Saminx22/text_embedding_model_14M"
config_path = hf_hub_download(repo_id=REPO_ID, filename="embedding_model_config.json")
weights_path = hf_hub_download(repo_id=REPO_ID, filename="embedding_model_weights.pth")

with open(config_path, 'r') as f: config = json.load(f)
# Corrected tokenizer loading: use subfolder argument
tokenizer = AutoTokenizer.from_pretrained(REPO_ID, subfolder="tokenizer")
model = EmbeddingModel(vocab_size=config['vocab_size'])
model.load_state_dict(torch.load(weights_path, map_location='cpu'))
model.eval()

def get_similarity(query, candidates_str):
    candidates = [c.strip() for c in candidates_str.split('\n') if c.strip()]
    if not candidates: return "Please enter some candidate sentences."
    
    with torch.no_grad():
        q_ids = tokenizer(query, padding='max_length', truncation=True, max_length=32, return_tensors='pt')['input_ids']
        q_emb = model(q_ids)
        c_ids = tokenizer(candidates, padding='max_length', truncation=True, max_length=32, return_tensors='pt')['input_ids']
        c_embs = model(c_ids)
        
        probs = torch.matmul(q_emb, c_embs.T).squeeze(0)
        results = sorted(zip(candidates, probs.tolist()), key=lambda x: x[1], reverse=True)
        
    return "\n".join([f"{score:.4f} | {text}" for text, score in results])

demo = gr.Interface(
    fn=get_similarity,
    inputs=[gr.Textbox(label="Query Sentence"), gr.Textbox(label="Candidate Sentences (one per line)", lines=5)],
    outputs=gr.Textbox(label="Similarity Scores"),
    title="14M Parameter Text Embedding Demo",
    description="Using the custom-trained model from Saminx22/text_embedding_model_14M"
)

demo.launch()
"""

# 3. Upload corrected app.py to the Space
with open("app.py", "w") as f:
    f.write(app_code.strip())

api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=space_id, repo_type="space")

print(f"Space updated at: https://huggingface.co/spaces/{space_id}")

In [ ]:
# 4. Create and upload requirements.txt to the Space
requirements_content = """
torch
transformers
gradio
huggingface_hub
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=space_id,
    repo_type="space"
)

print(f"requirements.txt uploaded to: https://huggingface.co/spaces/{space_id}/tree/main")